# Talk to your data

_Hand an agent a real CSV and ask in plain English. Get back a typed report — a
summary, a per-region table, and a chart — with no glue code._

⏱️ **~5 min** · Part of the [JAZ](https://jaz-lang.ai) tutorial series.

> Run the setup cells once, then work through the rest top to bottom. You only need an
> LLM API key.

## Setup

In [ ]:
# Install JAZ from PyPI. pandas and matplotlib already ship with Colab.
%pip install -q jaz-lang

In [ ]:
import os, getpass

# JAZ routes to providers through LiteLLM and reads each provider's standard
# API-key environment variable. Set at least one. In Colab you can also use the
# secrets manager (key icon in the left sidebar).
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')


In [ ]:
import jaz
from jaz.llm import LiteLLM

# Pick the model all agents use. The model id is a LiteLLM route, 'provider/model'.
# Swap to an Anthropic model (and set ANTHROPIC_API_KEY) if you prefer.
jaz.configure(llm=LiteLLM(model='openai/gpt-5-mini'))


## The problem

You're an analyst. Someone drops a CSV on your desk with a fuzzy question:
_"Which countries drove the most CO₂ growth this century, how do emissions split by
region, and what's the overall trend?"_

Normally that's an afternoon of pandas. With JAZ you hand the agent the DataFrame and
ask in prose — it writes and runs the pandas itself, then hands back exactly the
shape you asked for.

## Load some real data

We'll use [Our World in Data's CO₂ dataset](https://github.com/owid/co2-data) — real
countries, real years, ~50k rows. It loads straight from GitHub, no key required.

In [ ]:
import pandas as pd

# Pin to a stable path. Swap `master` for a commit SHA to freeze it forever.
CO2_URL = 'https://github.com/owid/co2-data/raw/master/owid-co2-data.csv'
co2 = pd.read_csv(CO2_URL)

print(f'{len(co2):,} rows x {co2.shape[1]} columns')
co2[['country', 'year', 'iso_code', 'co2', 'co2_growth_abs']].tail()

## Ask in prose, get back a typed report

Here's the whole idea:

1. **Declare the shape of the answer** as a dataclass. Fields can be anything — including
   a `DataFrame` and a matplotlib `Figure`.
2. **Pass the DataFrame straight in** as an ordinary keyword argument. Inside the agent's
   code it's just a variable named `co2`.
3. **Describe the task** in plain language.

No schema wrangling, no output-format prompt engineering. A `ValidateReturn` hook is
optional — `ReturnType` already guarantees the shape; the validator adds a couple of
sanity checks, and if it raises, the agent sees the error and tries again.


In [ ]:
from jaz import invoke
from jaz.hooks import ReturnType, ValidateReturn
from dataclasses import dataclass
from matplotlib.figure import Figure

@dataclass
class Report:
    summary: str             # a plain-language answer to the question
    by_region: pd.DataFrame  # emissions broken down by continent
    trend: Figure            # a chart of the global CO2 trend over time

def check(report: Report) -> None:
    assert report.summary.strip(), 'summary must not be empty'
    assert len(report.by_region) > 0, 'expected at least one region'

report = invoke(
    ReturnType(Report),
    ValidateReturn(check),
    task=(
        'Using the `co2` DataFrame, answer three things about CO2 emissions: '
        '(1) the 5 countries with the largest absolute increase in annual emissions '
        'since 2000; (2) a breakdown of the most recent year\'s emissions by continent; '
        'and (3) a line chart of global emissions over time. '
        'Tips: the `country` column mixes individual countries with aggregates such as '
        'continents and \'World\' (aggregates have an empty `iso_code`) — use the continent '
        'rows for the regional breakdown, and exclude aggregates when ranking countries.'
    ),
    co2=co2,
)


## See what it built

`report` is a real `Report` instance — typed fields you can keep working with.

In [ ]:
print(report.summary)
report.by_region

In [ ]:
report.trend  # renders inline; report.trend.savefig('trend.png') to export

## Why this matters

- **Any context, any shape** — you passed a ~50k-row DataFrame in as-is. No
  serialization, no chunking, no vector store.
- **Just pass Python** — `co2` is an ordinary keyword argument the agent used like any
  local variable.
- **Validated outputs** — you declared a `Report` and got a `Report`, checked before it
  reached you.

The agent wrote and ran the pandas and matplotlib itself, and you can inspect every step
it took. Read the [docs](https://jaz-lang.ai/docs) to go deeper.